# Cyber Sachet - Detailed Cell-by-Cell Implementation

Nepal Cyber Security and Law Assistant with Full Code Visibility

## Part 1: Environment Setup

In [ ]:
# Install required packages
!pip install -q openai chromadb python-dotenv pytest pytest-asyncio

In [ ]:
import os
import asyncio
from datetime import datetime
from typing import List, Dict, Any, Optional, Set
from collections import Counter
from dataclasses import dataclass
from dotenv import load_dotenv

print("Standard libraries imported")

In [ ]:
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print(f"API Key loaded (length: {len(api_key)})")
else:
    print("API Key not found!")

## Part 2: Data Models

In [ ]:
@dataclass
class SearchResult:
    """Search result from vector database."""
    content: str
    source: str
    doc_type: str
    relevance_score: float
    chunk_id: int
    metadata: Dict[str, Any]
    
    def to_dict(self) -> Dict[str, Any]:
        """Convert to dictionary."""
        return {
            "content": self.content,
            "source": self.source,
            "doc_type": self.doc_type,
            "relevance_score": self.relevance_score,
            "chunk_id": self.chunk_id,
            "metadata": self.metadata
        }

print("SearchResult model defined")

In [ ]:
@dataclass
class ToolCall:
    """Record of a tool call."""
    tool_name: str
    args: Dict[str, Any]
    result: Any
    duration_ms: float
    timestamp: str = None
    
    def __post_init__(self):
        if self.timestamp is None:
            self.timestamp = datetime.now().isoformat()

print("ToolCall model defined")

## Part 3: Database Utilities

In [ ]:
# Document type mapping
DOC_TYPE_MAPPING = {
    "cyber_awareness_guide": "awareness",
    "nepal_digital_security_act": "cyber_law",
    "nepal_information_technology_act": "cyber_law"
}

def get_doc_type(filename: str) -> str:
    """Get document type from filename."""
    file_key = filename.replace(".txt", "")
    return DOC_TYPE_MAPPING.get(file_key, "awareness")

print("✓ Document type mapper defined")

In [ ]:
def chunk_text(content: str, chunk_size: int = 500, chunk_overlap: int = 100) -> List[str]:
    """Split text into overlapping chunks."""
    chunks = []
    start = 0
    
    while start < len(content):
        end = start + chunk_size
        chunk = content[start:end]
        
        if end < len(content):
            last_period = chunk.rfind(". ")
            if last_period > chunk_size // 2:
                end = start + last_period + 1
                chunk = content[start:end]
        
        chunk_stripped = chunk.strip()
        if chunk_stripped:
            chunks.append(chunk_stripped)
        
        start = end - chunk_overlap
    
    return chunks

print("Text chunker defined")

In [ ]:
# File reading utilities
def read_text_file(filepath: str) -> str:
    """Read text file content."""
    with open(filepath, "r", encoding="utf-8") as f:
        return f.read()

def get_text_files(docs_folder: str) -> List[str]:
    """Get list of .txt files in a folder."""
    return [f for f in os.listdir(docs_folder) if f.endswith(".txt")]

print("✓ File reader utilities defined")

In [ ]:
def load_documents(collection, docs_folder: str = "docs/", 
                   chunk_size: int = 500, chunk_overlap: int = 100) -> Dict[str, Any]:
    """Load documents from folder into ChromaDB."""
    documents = []
    metadatas = []
    ids = []
    chunk_counter = 0
    
    stats = {
        "files_loaded": [],
        "total_chunks": 0,
        "doc_types": {}
    }
    
    text_files = get_text_files(docs_folder)
    
    for filename in text_files:
        filepath = os.path.join(docs_folder, filename)
        
        doc_type = get_doc_type(filename)
        
        content = read_text_file(filepath)
        chunks = chunk_text(content, chunk_size, chunk_overlap)
        
        file_key = filename.replace(".txt", "")
        for i, chunk in enumerate(chunks):
            documents.append(chunk)
            metadatas.append({
                "source": filename,
                "doc_type": doc_type,
                "chunk_id": i,
                "total_chunks": len(chunks)
            })
            ids.append(f"{file_key}_{i}")
            chunk_counter += 1
        
        stats["files_loaded"].append(filename)
        if doc_type not in stats["doc_types"]:
            stats["doc_types"][doc_type] = {"files": 0, "chunks": 0}
        stats["doc_types"][doc_type]["files"] += 1
        stats["doc_types"][doc_type]["chunks"] += len(chunks)
    
    if documents:
        collection.add(documents=documents, metadatas=metadatas, ids=ids)
    
    stats["total_chunks"] = chunk_counter
    return stats

print("Document loader defined")

## Part 4: Tool Components

In [ ]:
# Tool logger class
class ToolLogger:
    """Tracks tool calls for debugging."""
    
    def __init__(self, enable_logging: bool = True):
        self.enable_logging = enable_logging
        self.tool_call_history: List[ToolCall] = []
    
    def log_call(self, tool_name: str, args: Dict[str, Any], result: Any, duration_ms: float):
        """Log a tool call."""
        if self.enable_logging:
            self.tool_call_history.append(
                ToolCall(tool_name=tool_name, args=args, result=result, duration_ms=duration_ms)
            )
    
    def get_history(self) -> List[ToolCall]:
        """Get history of all tool calls."""
        return self.tool_call_history
    
    def clear_history(self):
        """Clear tool call history."""
        self.tool_call_history = []
    
    def get_stats(self) -> Dict[str, Any]:
        """Get statistics about tool usage."""
        if not self.tool_call_history:
            return {"total_calls": 0}
        
        tool_counts = Counter([call.tool_name for call in self.tool_call_history])
        avg_duration = sum(call.duration_ms for call in self.tool_call_history) / len(self.tool_call_history)
        
        return {
            "total_calls": len(self.tool_call_history),
            "tool_counts": dict(tool_counts),
            "average_duration_ms": round(avg_duration, 2),
            "total_duration_ms": sum(call.duration_ms for call in self.tool_call_history)
        }

print("✓ ToolLogger class defined")

In [ ]:
# Semantic search tool
async def semantic_search_tool(collection, query: str, n_results: int = 5,
                                doc_type_filter: Optional[str] = None,
                                min_relevance: float = 0.0) -> List[SearchResult]:
    """Perform semantic search on cyber security documents."""
    # Validate inputs
    n_results = min(max(1, n_results), 20)
    min_relevance = max(0.0, min(1.0, min_relevance))
    
    # Build query parameters
    query_params = {
        "query_texts": [query],
        "n_results": n_results
    }
    
    if doc_type_filter:
        if doc_type_filter not in ["cyber_law", "awareness"]:
            raise ValueError(f"Invalid doc_type_filter: {doc_type_filter}")
        query_params["where"] = {"doc_type": doc_type_filter}
    
    # Execute search
    results = await asyncio.to_thread(collection.query, **query_params)
    
    # Process results
    search_results = []
    if results['documents'] and results['documents'][0]:
        for i in range(len(results['documents'][0])):
            distance = results['distances'][0][i] if 'distances' in results else 0
            relevance_score = max(0, 1 - distance)
            
            if relevance_score >= min_relevance:
                search_results.append(SearchResult(
                    content=results['documents'][0][i],
                    source=results['metadatas'][0][i]['source'],
                    doc_type=results['metadatas'][0][i]['doc_type'],
                    relevance_score=round(relevance_score, 3),
                    chunk_id=results['metadatas'][0][i]['chunk_id'],
                    metadata=results['metadatas'][0][i]
                ))
    
    return search_results

print("✓ semantic_search_tool defined")

In [ ]:
# Laws search tool
async def search_laws_tool(collection, query: str, n_results: int = 3) -> List[SearchResult]:
    """Search only in cyber law documents."""
    return await semantic_search_tool(collection, query, n_results, "cyber_law")

print("✓ search_laws_tool defined")

In [ ]:
# Awareness search tool
async def search_awareness_tool(collection, query: str, n_results: int = 3) -> List[SearchResult]:
    """Search only in cyber awareness documents."""
    return await semantic_search_tool(collection, query, n_results, "awareness")

print("✓ search_awareness_tool defined")

In [ ]:
# Document sources tool
async def get_document_sources_tool(collection) -> Dict[str, Any]:
    """Get list of all indexed document sources."""
    all_data = await asyncio.to_thread(collection.get)
    
    sources = {}
    for meta in all_data['metadatas']:
        source = meta['source']
        doc_type = meta['doc_type']
        
        if source not in sources:
            sources[source] = {"type": doc_type, "chunks": 0}
        sources[source]["chunks"] += 1
    
    return {
        "total_documents": len(sources),
        "total_chunks": len(all_data['metadatas']),
        "sources": sources
    }

print("✓ get_document_sources_tool defined")

In [ ]:
# Penalty checking tool
async def check_penalty_tool(collection, crime_type: str) -> Dict[str, Any]:
    """Check penalties for specific cybercrimes."""
    search_query = f"penalty punishment {crime_type} fine imprisonment sentence"
    results = await search_laws_tool(collection, search_query, n_results=3)
    
    penalty_info = {
        "crime_type": crime_type,
        "found": len(results) > 0,
        "details": [],
        "sources": []
    }
    
    for result in results:
        content_lower = result.content.lower()
        if any(word in content_lower for word in ['penalty', 'punishment', 'fine', 'imprisonment', 'sentence']):
            penalty_info["details"].append(result.content)
            if result.source not in penalty_info["sources"]:
                penalty_info["sources"].append(result.source)
    
    penalty_info["found"] = len(penalty_info["details"]) > 0
    return penalty_info

print("✓ check_penalty_tool defined")

In [ ]:
# Tools wrapper class
class CyberSachetTools:
    """Tool collection for Cyber Sachet Agent."""
    
    def __init__(self, collection, embedding_function, enable_logging: bool = True):
        self.collection = collection
        self.embedding_function = embedding_function
        self.logger = ToolLogger(enable_logging)
    
    def get_tool_call_history(self):
        return self.logger.get_history()
    
    def clear_history(self):
        self.logger.clear_history()
    
    def get_tool_stats(self) -> Dict[str, Any]:
        return self.logger.get_stats()
    
    async def semantic_search_tool(self, query: str, n_results: int = 5,
                                    doc_type_filter: Optional[str] = None,
                                    min_relevance: float = 0.0) -> List[SearchResult]:
        start_time = datetime.now()
        results = await semantic_search_tool(self.collection, query, n_results, doc_type_filter, min_relevance)
        duration = (datetime.now() - start_time).total_seconds() * 1000
        self.logger.log_call("semantic_search_tool",
                            {"query": query, "n_results": n_results, "doc_type_filter": doc_type_filter,
                             "min_relevance": min_relevance}, results, duration)
        return results
    
    async def search_laws_tool(self, query: str, n_results: int = 3) -> List[SearchResult]:
        start_time = datetime.now()
        results = await search_laws_tool(self.collection, query, n_results)
        duration = (datetime.now() - start_time).total_seconds() * 1000
        self.logger.log_call("search_laws_tool", {"query": query, "n_results": n_results}, results, duration)
        return results
    
    async def search_awareness_tool(self, query: str, n_results: int = 3) -> List[SearchResult]:
        start_time = datetime.now()
        results = await search_awareness_tool(self.collection, query, n_results)
        duration = (datetime.now() - start_time).total_seconds() * 1000
        self.logger.log_call("search_awareness_tool", {"query": query, "n_results": n_results}, results, duration)
        return results
    
    async def get_document_sources_tool(self) -> Dict[str, Any]:
        start_time = datetime.now()
        result = await get_document_sources_tool(self.collection)
        duration = (datetime.now() - start_time).total_seconds() * 1000
        self.logger.log_call("get_document_sources_tool", {}, result, duration)
        return result
    
    async def check_penalty_tool(self, crime_type: str) -> Dict[str, Any]:
        start_time = datetime.now()
        result = await check_penalty_tool(self.collection, crime_type)
        duration = (datetime.now() - start_time).total_seconds() * 1000
        self.logger.log_call("check_penalty_tool", {"crime_type": crime_type}, result, duration)
        return result

print("✓ CyberSachetTools class defined")

## Part 5: Agent Components

In [ ]:
# Agent prompts
SYSTEM_PROMPT = """You are a helpful AI assistant specializing in cyber security awareness and Nepal's cyber laws. 
Your role is to provide accurate, clear, and well-cited information.

Guidelines:
- Always cite sources for your information using [Source: filename]
- Provide specific, actionable advice when possible
- Explain legal terms in simple language
- For legal questions, reference specific laws and sections
- For security questions, provide practical tips

Remember: You must cite your sources in your response."""

def get_user_message(context: str, question: str) -> str:
    """Build user message with context."""
    return f"""Based on the following context from our knowledge base, please answer the question.
IMPORTANT: You must cite sources in your answer using [Source: filename].

CONTEXT:
{context}

QUESTION: {question}

Please provide a clear, well-cited answer:"""

print("✓ Agent prompts defined")

In [ ]:
# Tool selector
def select_tools_for_query(question: str) -> List[str]:
    """Select appropriate tools based on question."""
    question_lower = question.lower()
    tools = []
    
    # Check for penalty-specific questions
    if any(word in question_lower for word in ['penalty', 'punishment', 'fine', 'imprisonment', 'sentence']):
        tools.append('check_penalty_tool')
    
    # Check for legal questions
    if any(word in question_lower for word in ['law', 'legal', 'act', 'penalty', 'crime', 'illegal', 'offense', 'punishment']):
        tools.append('search_laws_tool')
    
    # Check for awareness questions
    if any(word in question_lower for word in ['how to', 'protect', 'secure', 'safe', 'prevent', 'tips', 'best practice']):
        tools.append('search_awareness_tool')
    
    # Default to semantic search if no specific tools
    if not tools:
        tools.append('semantic_search_tool')
    
    return tools

def extract_crime_type(question: str) -> str:
    """Extract crime type from question."""
    question_lower = question.lower()
    
    crime_keywords = {
        'hacking': ['hack', 'hacking', 'hacker'],
        'phishing': ['phish', 'phishing'],
        'fraud': ['fraud', 'scam'],
        'data breach': ['data breach', 'data theft', 'data leak'],
        'identity theft': ['identity theft', 'identity fraud'],
        'ransomware': ['ransomware'],
        'malware': ['malware', 'virus', 'trojan'],
        'cyberbullying': ['cyberbully', 'cyber bully', 'online harassment']
    }
    
    for crime_type, keywords in crime_keywords.items():
        if any(keyword in question_lower for keyword in keywords):
            return crime_type
    
    return "cybercrime"

print("✓ Tool selector functions defined")

In [ ]:
async def execute_tools(tools_obj, selected_tools: List[str], user_question: str,
                        n_context_docs: int = 5, verbose: bool = True):
    """Execute selected tools and collect results."""
    search_results: List[SearchResult] = []
    tools_used = []
    
    for tool_name in selected_tools:
        if tool_name == 'check_penalty_tool':
            crime_type = extract_crime_type(user_question)
            if verbose:
                print(f"Checking penalties for: {crime_type}")
            penalty_info = await tools_obj.check_penalty_tool(crime_type)
            tools_used.append('check_penalty_tool')
        
        elif tool_name == 'search_laws_tool':
            if verbose:
                print("Searching legal documents...")
            results = await tools_obj.search_laws_tool(user_question, n_results=n_context_docs)
            search_results.extend(results)
            tools_used.append('search_laws_tool')
        
        elif tool_name == 'search_awareness_tool':
            if verbose:
                print("Searching awareness guides...")
            results = await tools_obj.search_awareness_tool(user_question, n_results=n_context_docs)
            search_results.extend(results)
            tools_used.append('search_awareness_tool')
        
        elif tool_name == 'semantic_search_tool':
            if verbose:
                print("Performing semantic search...")
            results = await tools_obj.semantic_search_tool(user_question, n_results=n_context_docs)
            search_results.extend(results)
            tools_used.append('semantic_search_tool')
    
    tools_used = list(dict.fromkeys(tools_used))
    
    if verbose and search_results:
        print(f"Retrieved {len(search_results)} relevant documents")
        for i, r in enumerate(search_results[:3], 1):
            print(f"  {i}. {r.source} (relevance: {r.relevance_score})")
    
    return search_results, tools_used

print("Query executor defined")

In [ ]:
# Context builder
def build_context(search_results: List[SearchResult]):
    """Build context from search results."""
    context_parts = []
    sources_used = set()
    
    for result in search_results:
        context_parts.append(
            f"[Source: {result.source}, Type: {result.doc_type}, Relevance: {result.relevance_score}]\n"
            f"{result.content}"
        )
        sources_used.add(result.source)
    
    context = "\n\n---\n\n".join(context_parts)
    return context, sources_used

print("✓ Context builder defined")

In [ ]:
async def generate_response(client, system_prompt: str, user_message: str,
                             model: str = "gpt-4o-mini", temperature: float = 0.7,
                             verbose: bool = True) -> str:
    """Generate response using OpenAI LLM."""
    if verbose:
        print("Generating response...\n")
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}
    ]
    
    response = await client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_tokens=1500
    )
    
    return response.choices[0].message.content

print("Response generator defined")

In [ ]:
# Statistics tracker
def calculate_stats(query_history: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Calculate statistics from query history."""
    if not query_history:
        return {"total_queries": 0}
    
    all_tools = []
    for query in query_history:
        all_tools.extend(query['tools_used'])
    
    tool_counts = Counter(all_tools)
    avg_context_docs = sum(q['num_context_docs'] for q in query_history) / len(query_history)
    avg_duration = sum(q['query_duration_ms'] for q in query_history) / len(query_history)
    
    all_sources = set()
    for query in query_history:
        all_sources.update(query['sources_used'])
    
    return {
        "total_queries": len(query_history),
        "tool_usage": dict(tool_counts),
        "average_context_docs": round(avg_context_docs, 2),
        "average_duration_ms": round(avg_duration, 2),
        "unique_sources_used": len(all_sources),
        "sources": sorted(list(all_sources))
    }

print("✓ Statistics tracker defined")

In [ ]:
class CyberSachetAgent:
    """Cyber Sachet Agent - AI Assistant for Cyber Security and Nepal Cyber Laws."""
    
    def __init__(self, tools: CyberSachetTools, async_client):
        from openai import AsyncOpenAI
        self.tools = tools
        self.client = async_client
        self.query_history: List[Dict[str, Any]] = []
        self.system_prompt = SYSTEM_PROMPT
    
    async def query(self, user_question: str, n_context_docs: int = 5,
                    verbose: bool = True, model: str = "gpt-4o-mini",
                    temperature: float = 0.7) -> Dict[str, Any]:
        """Query the agent with a question."""
        query_start_time = datetime.now()
        
        if verbose:
            print(f"\nProcessing query: '{user_question}'")
        
        selected_tools = select_tools_for_query(user_question)
        
        if verbose:
            print(f"Selected tools: {', '.join(selected_tools)}")
        
        search_results, tools_used = await execute_tools(
            self.tools, selected_tools, user_question, n_context_docs, verbose
        )
        
        context, sources_used = build_context(search_results)
        
        user_message = get_user_message(context, user_question)
        
        answer = await generate_response(
            self.client, self.system_prompt, user_message, model, temperature, verbose
        )
        
        result = {
            "question": user_question,
            "answer": answer,
            "sources_used": sorted(list(sources_used)),
            "tools_used": tools_used,
            "num_context_docs": len(search_results),
            "search_results": [r.to_dict() for r in search_results],
            "timestamp": datetime.now().isoformat(),
            "model": model,
            "query_duration_ms": (datetime.now() - query_start_time).total_seconds() * 1000
        }
        
        self.query_history.append(result)
        return result
    
    async def quick_query(self, user_question: str, **kwargs) -> str:
        """Quick query returning just the answer."""
        kwargs['verbose'] = kwargs.get('verbose', False)
        result = await self.query(user_question, **kwargs)
        return result["answer"]
    
    def get_query_history(self) -> List[Dict[str, Any]]:
        """Get history of all queries."""
        return self.query_history
    
    def clear_history(self):
        """Clear query history."""
        self.query_history = []
        self.tools.clear_history()
    
    def get_stats(self) -> Dict[str, Any]:
        """Get statistics about agent usage."""
        return calculate_stats(self.query_history)

print("CyberSachetAgent class defined")

## Part 6: Initialize Everything

In [ ]:
# Import OpenAI and ChromaDB
from openai import AsyncOpenAI
import chromadb
from chromadb.utils import embedding_functions

print("✓ Libraries imported")

In [ ]:
# Create OpenAI client
async_client = AsyncOpenAI(api_key=api_key)
print("✓ OpenAI client created")

In [ ]:
# Create ChromaDB client
chroma_client = chromadb.Client()
print("✓ ChromaDB client created")

In [ ]:
# Create embedding function
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=api_key,
    model_name="text-embedding-3-small"
)
print("✓ Embedding function created")

In [ ]:
# Get or create collection
collection = chroma_client.get_or_create_collection(
    name="cyber_sachet",
    embedding_function=openai_ef
)
print(f"✓ Collection ready: {collection.count()} documents")

In [ ]:
# Load documents if needed
if collection.count() == 0:
    print("Loading documents...")
    stats = load_documents(collection, docs_folder="docs/", chunk_size=500)
    print(f"✓ Loaded {stats['total_chunks']} chunks from {len(stats['files_loaded'])} files")
else:
    print(f"✓ Collection already populated")

In [ ]:
# Create tools
tools = CyberSachetTools(collection, openai_ef, enable_logging=True)
print("✓ Tools created")

In [ ]:
# Create agent
agent = CyberSachetAgent(tools, async_client)
print("✓ Agent created and ready!")

## Part 7: Test the Agent

In [ ]:
# Test query
result = await agent.query("What is phishing?", verbose=True, n_context_docs=3)

print("\n" + "="*70)
print("ANSWER:")
print("="*70)
print(result['answer'])
print("\n" + "="*70)
print(f"📚 Sources: {', '.join(result['sources_used'])}")
print(f"🛠️  Tools: {', '.join(result['tools_used'])}")
print("="*70)

In [ ]:
# Test legal query
result = await agent.query("What are the penalties for hacking in Nepal?", verbose=True)

print("\n" + "="*70)
print("ANSWER:")
print("="*70)
print(result['answer'])
print("\n" + "="*70)
print(f"📚 Sources: {', '.join(result['sources_used'])}")
print(f"🛠️  Tools: {', '.join(result['tools_used'])}")
print("="*70)

In [ ]:
# View statistics
stats = agent.get_stats()
print("Agent Statistics:")
print(f"  Total queries: {stats['total_queries']}")
print(f"  Tool usage: {stats['tool_usage']}")
print(f"  Avg docs: {stats['average_context_docs']}")
print(f"  Avg duration: {stats['average_duration_ms']:.2f}ms")

## Part 8: Interactive Function

In [ ]:
# Interactive query helper
async def ask(question: str):
    """Quick helper to ask questions."""
    result = await agent.query(question, verbose=True)
    print("\n" + "="*70)
    print("ANSWER:")
    print("="*70)
    print(result['answer'])
    print("\n" + "="*70)
    print(f"📚 Sources: {', '.join(result['sources_used'])}")
    print("="*70)

print("✓ Helper function ready. Usage: await ask('your question')")

In [ ]:
# Try it
await ask("How can I protect myself from ransomware?")

## Summary

All code is now visible cell-by-cell:
- ✅ 15 separate tool files
- ✅ All code shown in notebook cells
- ✅ Each cell has single responsibility
- ✅ Easy to understand and modify
- ✅ Fully functional agent

**Total Modules Created:**
- Tools: 6 files
- Database: 3 files
- Agent: 4 files
- Models: 2 files
- **Total: 15+ small, focused files**